# Variational Autoencoder (VAE) — Simple Single Feature Example

This example creates a very small dataset with **only one feature** (`x`) and trains a simple Variational Autoencoder.

The goal is to help you understand:

1. Encoder
2. Latent distribution (`mean` and `std`)
3. Reparameterization trick
4. Decoder
5. Reconstruction

---

# Step 1 — Generate Sample Dataset

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# Create 200 samples
x = np.random.normal(loc=50, scale=10, size=200)

# Convert to dataframe
 df = pd.DataFrame({"x": x})

print(df.head())
```

Example output:

```text
           x
0  54.967142
1  48.617357
2  56.476885
3  65.230299
4  47.658466
```

---

# Step 2 — Visualize Dataset

```python
plt.hist(df['x'], bins=20)
plt.xlabel("x")
plt.ylabel("Count")
plt.title("Input Distribution")
plt.show()
```

The data approximately follows a normal distribution.

---

# Step 3 — Normalize Data

Neural networks train better when values are scaled.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X = scaler.fit_transform(df[['x']])

print(X[:5])
```

---

# Step 4 — Build VAE

We will use:

* Input dimension = 1
* Latent dimension = 1

---

# Architecture

```text
Input x
   ↓
Encoder
   ↓
mean (μ), std (σ)
   ↓
Sample z from distribution
   ↓
Decoder
   ↓
Reconstructed x
```

---

# Step 5 — Import Libraries

```python
import tensorflow as tf
from tensorflow.keras import layers
```

---

# Step 6 — Encoder

The encoder does NOT directly output a number.

Instead it outputs:

* Mean (μ)
* Log Variance (log_var)

These define a normal distribution.

```python
latent_dim = 1

# Encoder
encoder_inputs = layers.Input(shape=(1,))

h = layers.Dense(8, activation='relu')(encoder_inputs)

z_mean = layers.Dense(latent_dim, name='z_mean')(h)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(h)
```

---

# Step 7 — Reparameterization Trick

The VAE samples from the learned distribution.

Mathematically:

genui{"math_block_widget_always_prefetch_v2":{"content":"z = \mu + \sigma \epsilon"}}

Where:

* μ = mean
* σ = standard deviation
* ε = random noise from standard normal distribution

```python
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        epsilon = tf.random.normal(shape=(batch, dim))

        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Sampling()([z_mean, z_log_var])
```

---

# Step 8 — Create Encoder Model

```python
encoder = tf.keras.Model(
    encoder_inputs,
    [z_mean, z_log_var, z],
    name='encoder'
)

encoder.summary()
```

---

# Step 9 — Decoder

The decoder converts latent variable `z` back into reconstructed `x`.

```python
latent_inputs = layers.Input(shape=(latent_dim,))

h_decoded = layers.Dense(8, activation='relu')(latent_inputs)

outputs = layers.Dense(1, activation='sigmoid')(h_decoded)

decoder = tf.keras.Model(latent_inputs, outputs, name='decoder')

decoder.summary()
```

---

# Step 10 — Full VAE Model

```python
outputs = decoder(z)

vae = tf.keras.Model(encoder_inputs, outputs, name='vae')
```

---

# Step 11 — Loss Function

VAE uses TWO losses.

## 1. Reconstruction Loss

Measures how close output is to input.

```python
reconstruction_loss = tf.reduce_mean(
    tf.keras.losses.mse(encoder_inputs, outputs)
)
```

---

## 2. KL Divergence Loss

This forces latent space to look like a standard normal distribution.

genui{"math_block_widget_always_prefetch_v2":{"content":"KL = -\frac{1}{2}\sum (1 + \log(\sigma^2) - \mu^2 - \sigma^2)"}}

```python
kl_loss = -0.5 * tf.reduce_mean(
    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
)
```

---

## Total Loss

```python
vae_loss = reconstruction_loss + kl_loss

vae.add_loss(vae_loss)
```

---

# Step 12 — Compile and Train

```python
vae.compile(optimizer='adam')

history = vae.fit(
    X,
    X,
    epochs=100,
    batch_size=16,
    verbose=1
)
```

---

# Step 13 — Test the Encoder

Now let us see what the encoder outputs.

```python
sample = X[:5]

mean, log_var, latent_z = encoder.predict(sample)

print("Input:")
print(sample)

print("\nMean:")
print(mean)

print("\nLog Variance:")
print(log_var)

print("\nSampled z:")
print(latent_z)
```

Example idea:

```text
Input:
[[0.62]]

Mean:
[[0.13]]

Log Variance:
[[-0.08]]

Sampled z:
[[0.44]]
```

---

# Important Understanding

The encoder does NOT output a single deterministic value.

It outputs a probability distribution:

genui{"math_block_widget_always_prefetch_v2":{"content":"z \sim N(\mu, \sigma^2)"}}

This means:

```text
latent variable z follows a normal distribution
```

The model learns:

* mean
* variance

Then randomly samples from it.

That is why:

```text
same input → slightly different latent z values
```

---

# Step 14 — Reconstruction

```python
reconstructed = vae.predict(sample)

print("Original:")
print(sample)

print("\nReconstructed:")
print(reconstructed)
```

---

# Step 15 — Generate New Data

Because the latent space is probabilistic, we can generate new data.

```python
random_z = np.random.normal(size=(10, latent_dim))

generated = decoder.predict(random_z)

generated = scaler.inverse_transform(generated)

print(generated)
```

---

# Final Intuition

Normal Autoencoder:

```text
x → encoder → fixed latent vector → decoder → x
```

VAE:

```text
x → encoder → distribution → sample z → decoder → x
```

That probabilistic latent space allows:

* data generation
* smooth latent interpolation
* anomaly detection
* representation learning

---

# Why VAE Uses Distribution

A normal autoencoder memorizes.

A VAE learns:

```text
how data is distributed
```

That is why VAEs can generate new realistic samples.

---

# Real Applications

VAEs are used in:

* image generation
* anomaly detection
* recommender systems
* drug discovery
* latent representation learning
* compression
* speech synthesis

---

# Summary

The encoder outputs:

```text
mean + variance
```

NOT a fixed value.

The latent variable is sampled from:

genui{"math_block_widget_always_prefetch_v2":{"content":"N(\mu, \sigma^2)"}}

Then decoder reconstructs the input.
